In [ ]:
from google.colab import userdata
import os

# Securely fetch the key from Colab's secret manager
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
print("✅ API Key loaded successfully!")

✅ API Key loaded successfully!


In [ ]:
 #1. Install the Groq integration (run this once)
!pip install -q langchain-groq

# 2. Import Groq and set up the key
from langchain_groq import ChatGroq
from google.colab import userdata
import os

# Fetch the Groq key securely from Colab
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# 3. Initialize the Groq LLM (This replaces ChatOpenAI)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)

# Sanity check: This should print "ChatGroq", NOT "ChatOpenAI"
print("✅ LLM Initialized:", type(llm).__name__)

✅ LLM Initialized: ChatGroq


In [ ]:
from google.colab import userdata
import os

# Change 'secretName' to the EXACT name you used in the Colab Secrets UI
# If you named it "OPENAI_API_KEY" in the UI, use that here:
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

print("✅ API Key loaded successfully!")

✅ API Key loaded successfully!


In [ ]:
!pip install -q langchain-openai langgraph langchain

In [ ]:
# ==========================================
# 1. INSTALL & IMPORTS
# ==========================================
!pip install -q langchain-groq langgraph

from typing import TypedDict, Annotated, List
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from google.colab import userdata
import os

# ==========================================
# 2. INITIALIZE GROQ LLM
# ==========================================
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')


llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.3)

print(f"✅ LLM Initialized: {type(llm).__name__}")

# ==========================================
# 3. DEFINE STATE & AGENT NODES
# ==========================================
class AgentState(TypedDict):
    headline: str
    perspective: str
    messages: Annotated[List, add_messages]
    final_sentiment: str
    rationale: str

def advocate_node(state: AgentState):
    prompt = f"You are the Stakeholder Advocate. Your perspective: '{state['perspective']}'.\nHeadline: '{state['headline']}'.\nArgue why this headline evokes a strong positive or negative sentiment for this persona."
    return {"messages": [{"role": "assistant", "content": f"**Advocate**: {prompt}"}]}

def context_node(state: AgentState):
    prompt = f"You are the Context Expert. Provide objective, real-world background context about this headline: '{state['headline']}'. Ignore the persona; stick to facts."
    return {"messages": [{"role": "assistant", "content": f"**Context**: {prompt}"}]}

def skeptic_node(state: AgentState):
    prompt = f"You are the Skeptic. Review the Advocate's claim about '{state['headline']}'. Challenge any logical leaps or biases. Is the text itself actually negative, or is the persona projecting?"
    return {"messages": [{"role": "assistant", "content": f"**Skeptic**: {prompt}"}]}

def supervisor_node(state: AgentState):
    # NOTE: Using msg.content (dot notation) because LangGraph converts dicts to Message objects
    debate_history = "\n".join([msg.content for msg in state["messages"]])

    prompt = f"""You are the Judge. Review this debate about the headline: '{state['headline']}' from the perspective: '{state['perspective']}'.
    Debate History:
    {debate_history}

    Output strictly in this format:
    Sentiment Score: [Float from -1.0 to 1.0]
    Rationale: [2-sentence explanation]
    """
    response = llm.invoke(prompt) # This now uses Groq!
    return {"final_sentiment": response.content, "rationale": response.content}

# ==========================================
# 4. BUILD & COMPILE THE GRAPH
# ==========================================
workflow = StateGraph(AgentState)
workflow.add_node("advocate", advocate_node)
workflow.add_node("context", context_node)
workflow.add_node("skeptic", skeptic_node)
workflow.add_node("supervisor", supervisor_node)

workflow.set_entry_point("advocate")
workflow.add_edge("advocate", "context")
workflow.add_edge("context", "skeptic")
workflow.add_edge("skeptic", "supervisor")
workflow.add_edge("supervisor", END)

app = workflow.compile()
print("✅ Graph compiled successfully!")

# ==========================================
# 5. RUN THE DEBATE
# ==========================================
initial_state = {
    "headline": "Giannis sits out third consecutive game amid load management concerns.",
    "perspective": "I am a devoted Milwaukee Bucks fan who pays for tickets to see him play.",
    "messages": []
}

print("\n🔄 Running Agentic Debate...\n")
result = app.invoke(initial_state)

print("🏆 FINAL OUTPUT:")
print(result["final_sentiment"])

✅ LLM Initialized: ChatGroq
✅ Graph compiled successfully!

🔄 Running Agentic Debate...

🏆 FINAL OUTPUT:
Sentiment Score: -0.78  
Rationale: For a ticket‑paying Bucks fan, the headline signals disappointment and frustration because it confirms another missed opportunity to see Giannis play, directly affecting their game‑day experience. The emphasis on “load management concerns” further suggests the team is prioritizing future health over the fan’s immediate enjoyment, reinforcing a strongly negative reaction.


In [ ]:
import os
import requests

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"
headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

if response.status_code == 200:
    models = [model['id'] for model in response.json()['data']]
    print("✅ AVAILABLE GROQ MODELS FOR YOUR KEY:")
    for m in models:
        print(f"  - {m}")

    # Automatically pick the first 'llama' model found
    llama_models = [m for m in models if 'llama' in m.lower()]
    if llama_models:
        print(f"\n💡 Recommended: Update your code to use model='{llama_models[0]}'")
else:
    print(f"❌ Error fetching models: {response.status_code} - {response.text}")

✅ AVAILABLE GROQ MODELS FOR YOUR KEY:
  - canopylabs/orpheus-v1-english
  - canopylabs/orpheus-arabic-saudi
  - openai/gpt-oss-120b
  - openai/gpt-oss-safeguard-20b
  - meta-llama/llama-prompt-guard-2-86m
  - whisper-large-v3
  - qwen/qwen3.8-27b
  - meta-llama/llama-prompt-guard-2-22m
  - whisper-large-v3-turbo
  - openai/gpt-oss-20b
  - allam-2-7b

💡 Recommended: Update your code to use model='meta-llama/llama-prompt-guard-2-86m'
